# Exploration des Données - Phase 1

**Objectif :** Analyser les datasets Phase 1 pour préparer le pipeline ETL

**Approche scalable :**
- Code générique réutilisable pour Phase 2
- Pandas pour traiter 150 communes → extensible à 35K
- Fonctions modulaires pour chaque type d'analyse
- Optimisation mémoire avec chunks si nécessaire

**Plan d'exploration :**
1. Configuration et imports
2. Exploration systématique de chaque dataset
3. Analyse des clés de jointure
4. Qualité des données (valeurs manquantes, doublons)
5. Statistiques descriptives
6. Synthèse pour ETL

---

## 1. Configuration et Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports chargés (pandas, numpy)")

In [ ]:
# Options d'affichage pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Configuration pandas optimisée pour exploration")

In [ ]:
# === CONFIGURATION INLINE - Architecture Medallion ===
from pathlib import Path

# Détection automatique de la racine du projet
notebook_dir = Path.cwd()
if notebook_dir.name == "communes":
    PROJECT_ROOT = notebook_dir.parent.parent
else:
    PROJECT_ROOT = notebook_dir

# Chemins des données
DATA_BRONZE = PROJECT_ROOT / "data" / "bronze"

# Paramètres métier
DEPARTEMENTS_PETITE_COURONNE = ["75", "92", "93", "94"]
COMMUNES_PETITE_COURONNE_COUNT = 144

print(f"Bronze : {DATA_BRONZE}")
print(f"Périmètre : {chr(44).join(DEPARTEMENTS_PETITE_COURONNE)} ({COMMUNES_PETITE_COURONNE_COUNT} communes)")


## 2. Fonctions Utilitaires (Réutilisables pour Phase 2)

In [ ]:
def explore_dataset(df, name, show_sample=True, n_rows=5):
    """
    Exploration systématique d'un DataFrame pandas.
    Fonction générique réutilisable pour tous les datasets.
    
    Args:
        df: DataFrame pandas
        name: Nom du dataset
        show_sample: Afficher échantillon
        n_rows: Nombre de lignes à afficher
    """
    print(f"\n{'='*60}")
    print(f"📊 DATASET: {name}")
    print(f"{'='*60}\n")
    
    # Dimensions
    count = len(df)
    n_cols = len(df.columns)
    print(f"📏 Dimensions : {count:,} lignes × {n_cols} colonnes")
    
    # Types de données
    print(f"\n🏗️  Types de colonnes :")
    print(df.dtypes)
    
    # Info mémoire
    memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"\n💾 Mémoire utilisée : {memory_mb:.2f} MB")
    
    # Échantillon
    if show_sample:
        print(f"\n👁️  Échantillon ({n_rows} premières lignes) :")
        print(df.head(n_rows))
    
    return count, n_cols

print("✅ Fonction explore_dataset() définie")

In [ ]:
def analyze_missing_values(df, name):
    """
    Analyse des valeurs manquantes par colonne.
    Scalable pour grands datasets.
    """
    print(f"\n🔍 Valeurs manquantes - {name}")
    print("-" * 60)
    
    total_rows = len(df)
    
    # Calculer % manquants pour chaque colonne
    missing_data = df.isnull().sum()
    missing_pct = (missing_data / total_rows) * 100
    
    missing_stats = pd.DataFrame({
        'Colonne': missing_data.index,
        'Manquants': missing_data.values,
        'Pourcentage': missing_pct.values
    })
    
    # Filtrer uniquement colonnes avec valeurs manquantes
    missing_stats = missing_stats[missing_stats['Manquants'] > 0].sort_values(
        'Pourcentage', ascending=False
    )
    
    if len(missing_stats) > 0:
        print(f"\n{'Colonne':<30} {'Manquants':>12} {'%':>8}")
        print("-" * 60)
        for _, row in missing_stats.iterrows():
            print(f"{row['Colonne']:<30} {int(row['Manquants']):>12,} {row['Pourcentage']:>7.2f}%")
    else:
        print("✅ Aucune valeur manquante détectée")
    
    return missing_stats

print("✅ Fonction analyze_missing_values() définie")

In [ ]:
def filter_petite_couronne(df, dept_col='dept'):
    """
    Filtre pour Petite Couronne.
    Paramètre dept_col permet de s'adapter à différents noms de colonnes.
    
    Args:
        df: DataFrame avec colonne département
        dept_col: Nom de la colonne département
    """
    if dept_col not in df.columns:
        print(f"⚠️  Colonne '{dept_col}' non trouvée. Colonnes disponibles: {list(df.columns)}")
        return df
    
    df_filtered = df[df[dept_col].isin(DEPARTEMENTS_PETITE_COURONNE)].copy()
    count_before = len(df)
    count_after = len(df_filtered)
    
    print(f"🎯 Filtrage Petite Couronne :")
    print(f"   Avant : {count_before:,} lignes")
    print(f"   Après : {count_after:,} lignes ({count_after/count_before*100:.1f}%)")
    
    return df_filtered

print("✅ Fonction filter_petite_couronne() définie")

## 3. Exploration Dataset par Dataset

### 3.1 Référentiel Communes (Clé de Jointure)

Le dossier `referentiels_cog/` contient plusieurs millésimes :
- `referentiel_communes_2023.csv`
- `referentiel_communes_2024.csv` (utilisé pour les jointures)

La colonne `COM` est le code INSEE à 5 caractères, clé primaire universelle.

In [ ]:
# Charger referentiel communes (millésime 2024 — COG en vigueur)
# Dossier referentiels_cog/ contient plusieurs millésimes : 2023 et 2024
# On utilise 2024 pour les jointures (millésime le plus récent)
df_communes = pd.read_csv(
    DATA_BRONZE / "referentiels_cog" / "referentiel_communes_2024.csv",
    sep=","
)

explore_dataset(df_communes, "Référentiel Communes COG 2024", n_rows=3)

In [ ]:
# Filtrer Petite Couronne en créant colonne dept depuis code INSEE
df_communes['dept'] = df_communes['COM'].astype(str).str[:2]

df_communes_pc = filter_petite_couronne(df_communes, dept_col="dept")

print(f"\n📊 Répartition communes Petite Couronne :")
print(df_communes_pc.groupby('dept').size().sort_index())

In [ ]:
# Colonnes clés pour jointures
print("🔑 Colonnes clés identifiées :")
print(f"   - COM (Code INSEE) : clé primaire pour jointures")
print(f"   - LIBELLE : nom commune")
print(f"   - dept : département extrait du code INSEE")

# Sauvegarder liste codes INSEE Petite Couronne pour filtres futurs
codes_insee_pc = df_communes_pc['COM'].tolist()
print(f"\n✅ {len(codes_insee_pc)} codes INSEE Petite Couronne identifiés")

### 3.2 Élections Agrégées 1999-2024 (Variable Cible)

In [ ]:
# Charger élections (attention : fichier volumineux 2.2 GB)
print("⏳ Chargement élections agrégées (peut prendre 1-2 minutes)...")
print("💡 Utilisation de chunksize pour optimiser la mémoire...")

# Lire en chunks pour connaître la structure d'abord
# CORRECTION : ajouter sep=';' car fichier délimité par point-virgule
chunk_iterator = pd.read_csv(
    DATA_BRONZE / "elections_agregees_1999_2024.csv",
    sep=";",  # Fichier délimité par point-virgule
    chunksize=10000,
    nrows=10000,  # Juste un échantillon pour commencer
    dtype={'code_commune': str, 'code_departement': str}  # Codes INSEE en string
)

df_elections_sample = next(chunk_iterator)

print(f"\n✅ Échantillon chargé pour exploration")
explore_dataset(df_elections_sample, "Élections Agrégées (échantillon)", show_sample=False)

In [ ]:
# Échantillon aléatoire pour performance
print("👁️  Échantillon aléatoire (5 lignes) :")
print(df_elections_sample.sample(n=min(5, len(df_elections_sample))))

In [ ]:
# Analyser structure : types d'élections, années, niveaux géographiques
if 'election_type' in df_elections_sample.columns:
    print("📊 Types d'élections (échantillon) :")
    print(df_elections_sample['election_type'].value_counts())
else:
    print("ℹ️  Colonne 'election_type' non trouvée")
    print(f"Colonnes disponibles : {list(df_elections_sample.columns[:10])}")

if 'year' in df_elections_sample.columns:
    print("\n📅 Années couvertes (échantillon) :")
    print(df_elections_sample['year'].value_counts().sort_index())
elif 'annee' in df_elections_sample.columns:
    print("\n📅 Années couvertes (échantillon) :")
    print(df_elections_sample['annee'].value_counts().sort_index())

In [ ]:
# Identifier colonne code commune pour filtrage
print("🔍 Colonnes géographiques disponibles :")
geo_cols = [col for col in df_elections_sample.columns if 'code' in col.lower() or 'commune' in col.lower() or 'insee' in col.lower()]
print(geo_cols)

# Afficher échantillon pour comprendre structure
if geo_cols:
    print("\n👁️  Échantillon colonnes géo :")
    print(df_elections_sample[geo_cols].head(5))
else:
    print("\n⚠️  Aucune colonne géographique évidente détectée")
    print(f"Colonnes disponibles : {list(df_elections_sample.columns[:15])}")

In [ ]:
# TODO: Ajuster filtrage selon colonne identifiée ci-dessus
# Exemple générique (à adapter selon résultat précédent):
# df_elections_pc = df_elections_sample[df_elections_sample['code_commune'].isin(codes_insee_pc)]

print("💡 Note : Filtrage Petite Couronne à ajuster après identification colonne code commune")
print("💡 Pour traitement complet, charger en chunks et filtrer progressivement")

### 3.3 Revenus par Commune

In [ ]:
# Charger revenus
# CORRECTION : ajouter sep=';' car fichier délimité par point-virgule
df_revenus = pd.read_csv(
    DATA_BRONZE / "revenus_commune.csv",
    sep=";"  # Fichier délimité par point-virgule
)

explore_dataset(df_revenus, "Revenus par Commune", n_rows=3)

In [ ]:
# Analyser valeurs manquantes
analyze_missing_values(df_revenus, "Revenus")

In [ ]:
# Identifier colonne code commune et filtrer
print("🔍 Colonnes disponibles :")
print(df_revenus.columns.tolist())

# Chercher colonne avec code INSEE/commune
code_cols = [col for col in df_revenus.columns if 'code' in col.lower() or 'insee' in col.lower() or 'codgeo' in col.lower()]
print(f"\n🔑 Colonnes codes potentielles : {code_cols}")

# TODO: Adapter selon nom réel de la colonne code INSEE
# Exemple : df_revenus_pc = df_revenus[df_revenus['CODGEO'].isin(codes_insee_pc)]

### 3.4 Population Historique 1968-2022

In [ ]:
# Lister fichiers extraits
pop_dir = DATA_BRONZE / "population_historique_1968_2022"
pop_files = list(pop_dir.glob("*"))

print(f"📂 Fichiers dans {pop_dir.name}/ :")
for f in pop_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   • {f.name} ({size_mb:.2f} MB)")

In [ ]:
# Charger fichier population historique 1968-2022
# Structure feuille COM_2022 :
#   lignes 0-10  : metadonnees INSEE
#   ligne  11    : en-têtes multi-niveaux (DIPLOME / SEXE / AGE)
#   ligne  15    : labels lisibles (ex: 'Aucun diplôme\nHommes\n16 à 24 ans\nRP2022')
#   ligne  16    : codes courts (RR, DR, CR, dpx_rec0s1age2_rec1rpop2022...)
#   ligne  17+   : données communes
# => skiprows=16 pour avoir les codes courts comme en-têtes, données dès la 1ère ligne

data_files = [f for f in pop_files if f.suffix in ['.xlsx', '.xls'] and not f.name.startswith('.')]

if data_files:
    main_file = data_files[0]
    print(f"Fichier : {main_file.name}")
    df_population = pd.read_excel(
        main_file,
        sheet_name='COM_2022',
        skiprows=16,
        engine='openpyxl',
        nrows=5
    )
    print(f"Shape echantillon : {df_population.shape}")
    print(f"Colonnes ({len(df_population.columns)}) : {df_population.columns.tolist()[:8]} ...")
    print(df_population.iloc[:3, :6].to_string())
else:
    print("Aucun fichier XLSX trouve dans population_historique_1968_2022/")

### 3.5 Diplômes et Formation 2022

In [ ]:
# Lister fichiers extraits
diplomes_dir = DATA_BRONZE / "diplomes_formation_2022"
diplomes_files = list(diplomes_dir.glob("*"))

print(f"📂 Fichiers dans {diplomes_dir.name}/ :")
for f in diplomes_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   • {f.name} ({size_mb:.2f} MB)")

In [ ]:
# Charger diplômes (XLSX)
print("💡 Recherche du fichier principal...")

data_files = [f for f in diplomes_files if f.suffix in ['.xlsx', '.xls', '.csv'] and not f.name.startswith('.')]

if data_files:
    main_file = data_files[0]
    print(f"📄 Fichier trouvé : {main_file.name}")
    
    if main_file.suffix in ['.xlsx', '.xls']:
        print("⏳ Chargement Excel avec openpyxl...")
        # Charger juste un échantillon (skip 4 metadata rows)
        df_diplomes = pd.read_excel(main_file, engine='openpyxl', skiprows=4, nrows=10)
        print(f"✅ Échantillon chargé (10 lignes)")
        print(df_diplomes.head())
        print(f"\n📋 Colonnes : {df_diplomes.columns.tolist()}")
    elif main_file.suffix == '.csv':
        df_diplomes = pd.read_csv(main_file, nrows=10)
        print(f"✅ Échantillon chargé (10 lignes)")
        print(df_diplomes.head())
else:
    print("⚠️  Aucun fichier de données trouvé")

### 3.6 CSP des Actifs 25-54 ans

In [ ]:
# Lister fichiers extraits
csp_dir = DATA_BRONZE / "csp_actifs_2554"
csp_files = list(csp_dir.glob("*"))

print(f"📂 Fichiers dans {csp_dir.name}/ :")
for f in csp_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   • {f.name} ({size_mb:.2f} MB)")

In [ ]:
# Charger fichier CSP actifs 25-54 ans
# Structure feuille COM_2022 :
#   lignes 0-14 : metadonnees INSEE
#   ligne  15   : codes courts (RR, DR, CR, LIBELLE, csx_rec1...)
#   ligne  16+  : donnees communes
# => skiprows=15 (1 ligne de moins que pop-16ans-dipl6822.xlsx)

data_files = [f for f in csp_files if f.suffix in [".xlsx", ".xls"] and not f.name.startswith(".")]

if data_files:
    main_file = data_files[0]
    print(f"Fichier : {main_file.name}")
    df_csp = pd.read_excel(
        main_file,
        sheet_name="COM_2022",
        skiprows=15,
        engine="openpyxl",
        nrows=5
    )
    print(f"Shape echantillon : {df_csp.shape}")
    print(f"Colonnes ({len(df_csp.columns)}) : {df_csp.columns.tolist()[:8]} ...")
    print(df_csp.iloc[:3, :6].to_string())
else:
    print("Aucun fichier XLSX trouve dans csp_actifs_2554/")


### 3.7 Naissances 2008-2024

Fichier : `naissances_2008_2024/DS_ETAT_CIVIL_NAIS_COMMUNES_data.csv`

Source : INSEE — Etat civil, naissances par commune.

In [ ]:
# Charger naissances 2008-2024 (echantillon)
naissances_file = DATA_BRONZE / "naissances_2008_2024" / "DS_ETAT_CIVIL_NAIS_COMMUNES_data.csv"

if naissances_file.exists():
    df_naissances = pd.read_csv(naissances_file, sep=';', nrows=5)
    print(f"Colonnes : {df_naissances.columns.tolist()}")
    print(df_naissances.head().to_string())
else:
    print(f"Fichier non trouve : {naissances_file}")
    print("Telecharger via 01_data_download.ipynb")

### 3.8 Décès 2008-2024

Fichier : `deces_2008_2024/DS_ETAT_CIVIL_DECES_COMMUNES_data.csv`

Source : INSEE — Etat civil, décès par commune.

In [ ]:
# Charger deces 2008-2024 (echantillon)
deces_file = DATA_BRONZE / "deces_2008_2024" / "DS_ETAT_CIVIL_DECES_COMMUNES_data.csv"

if deces_file.exists():
    df_deces = pd.read_csv(deces_file, sep=';', nrows=5)
    print(f"Colonnes : {df_deces.columns.tolist()}")
    print(df_deces.head().to_string())
else:
    print(f"Fichier non trouve : {deces_file}")
    print("Telecharger via 01_data_download.ipynb")

## 4. Analyse des Jointures Possibles

### Stratégie de jointure

In [ ]:
print("🔗 Stratégie de jointure identifiée :")
print("""
Clé primaire : Code INSEE commune

Schéma de jointure :
    
    referentiel_communes (COM)
         ├─ LEFT JOIN elections (code_commune)
         ├─ LEFT JOIN revenus (CODGEO)
         ├─ LEFT JOIN population (code_commune)
         ├─ LEFT JOIN diplomes (CODGEO)
         └─ LEFT JOIN csp (CODGEO)

Type de jointure : LEFT JOIN
Raison : Garder toutes communes Petite Couronne même si données manquantes

""")

print("💡 À valider dans notebook ETL :")
print("   - Noms exacts colonnes codes INSEE par dataset")
print("   - Gestion des doublons (agrégations temporelles)")
print("   - Traitement valeurs manquantes")

## 5. Synthèse de l'Exploration

### Points clés identifiés

In [ ]:
print("""
SYNTHESE EXPLORATION
========================================

Datasets explores :
   1. Referentiel communes COG 2024  (referentiels_cog/referentiel_communes_2024.csv)
   2. Elections agregees 1999-2024   (elections_agregees_1999_2024.csv)
   3. Revenus par commune            (revenus_commune.csv)
   4. Population historique 1968-2022 (pop-16ans-dipl6822.xlsx, sheet COM_2022, skiprows=16)
   5. Diplomes et formation 2022     (base-cc-diplomes-formation-2022.xlsx)
   6. CSP actifs 25-54 ans           (pop-act2554-csp-cd-6822.xlsx, sheet COM_2022, skiprows=16)
   7. Naissances 2008-2024           (DS_ETAT_CIVIL_NAIS_COMMUNES_data.csv)
   8. Deces 2008-2024                (DS_ETAT_CIVIL_DECES_COMMUNES_data.csv)

Cle de jointure : Code INSEE commune (colonne COM / CR)

Prochaines etapes :
   -> petite_couronne/01_etl_bronze_to_postgres.ipynb
   -> france/01_etl_bronze_to_postgres.ipynb
""")

In [ ]:
# Libérer la mémoire
del df_elections_sample
import gc
gc.collect()

print("✅ Mémoire libérée")

---

## Notes pour la Suite

### Fichiers XLSX INSEE (Population historique et CSP)

Les deux fichiers XLSX (`pop-16ans-dipl6822.xlsx` et `pop-act2554-csp-cd-6822.xlsx`) ont la même structure :
- Feuille active par défaut : `Présentation` (vide)
- Données communales : feuilles `COM_1968`, `COM_1975`, ..., `COM_2022`
- 16 lignes de métadonnées avant les données réelles
- **Paramètres corrects** : `sheet_name='COM_2022'`, `skiprows=16`, `engine='openpyxl'`

### Prochaines étapes

Les notebooks ETL chargent les fichiers complets et écrivent en PostgreSQL (schémas `silver` puis `gold`) :
- `petite_couronne/01_etl_bronze_to_postgres.ipynb` — ETL Petite Couronne
- `france/01_etl_bronze_to_postgres.ipynb` — ETL France entière

Silver et Gold sont entièrement dans PostgreSQL, pas de fichiers Parquet locaux.